# iNaturalist *Acacia* observations — Coimbra district

**Purpose.** Look at the citizen-science data on their own, before any
modelling: how many observations, when, where, how clustered, with
what quality, and by whom.

**Data.** Every *Acacia* observation in the iNaturalist place
"Coimbra, PT" (the Coimbra **district**), as downloaded
(`data/raw/citizen_science/`, see `data/README.md`). Nothing is
filtered here.

## 0. Setup

Loads the CSV and turns each observation into a point in EPSG:3763
(PT-TM06, in metres), so distances and grid cells are in metres.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
inat = pd.read_csv(ROOT / "data" / "raw" / "citizen_science" / "inaturalist_acacia_coimbra_pilot.csv")
inat["observed_on"] = pd.to_datetime(inat["observed_on"], errors="coerce")
inat = gpd.GeoDataFrame(inat, geometry=gpd.points_from_xy(inat.longitude, inat.latitude), crs=4326).to_crs(3763)
print(f"{len(inat):,} observations")

## 1. What comes with each observation

One row per observation. The columns that matter later:
- `observed_on`: the date;
- `taxon_name`, `taxon_rank`: the species (or only the genus);
- `latitude`, `longitude`, `positional_accuracy` (metres, set by the
  observer), `coordinates_obscured` (location deliberately moved);
- `quality_grade`: `research` (identity confirmed by the community),
  `needs_id` or `casual`;
- `user_login`: the observer.

There is **no absence, abundance or cover**: each row only says
"*Acacia* was seen here".

In [ ]:
(inat.drop(columns="geometry").isna().mean() * 100).round(1).to_frame("pct_missing")

## 2. When

Observations per year and per month. The month matters because
*A. dealbata* flowers in winter (yellow, easy to spot): people may
record it mostly then.

In [ ]:
year = inat["observed_on"].dt.year
print("from", inat["observed_on"].min().date(), "to", inat["observed_on"].max().date())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
year.value_counts().sort_index().loc[2010:].plot.bar(ax=axes[0], color="tab:green", title="Observations per year (from 2010)")
inat["observed_on"].dt.month.value_counts().sort_index().plot.bar(ax=axes[1], color="tab:green", title="Observations per month")
plt.tight_layout()

## 3. Several observations in the same place

A 10 m cell is one satellite pixel: observations sharing one become
the same training example. Shown for 10 m, 100 m and 1 km cells.

In [ ]:
def cell(gdf, size):
    return (gdf.geometry.x // size).astype(int).astype(str) + "_" + (gdf.geometry.y // size).astype(int).astype(str)

pd.DataFrame([dict(cell=f"{s} m", cells_with_obs=cell(inat, s).nunique(),
                   pct_obs_sharing_a_cell=round(100 * cell(inat, s).duplicated(keep=False).mean(), 1))
              for s in [10, 100, 1000]])

## 4. Where: dense areas and empty areas

Observations per 1 km cell (log colour scale; empty cells blank). An
empty cell means nobody recorded *Acacia* there, **not** that there
is none.

In [ ]:
counts = cell(inat, 1000).value_counts()
for k in [1, 10, 50]:
    print(f"Top {k:>2} cells (of {len(counts)}) hold {counts.iloc[:k].sum() / len(inat):.1%} of observations")

x, y = inat.geometry.x // 1000, inat.geometry.y // 1000
fig, ax = plt.subplots(figsize=(8, 7))
h = ax.hist2d(x, y, bins=[np.arange(x.min(), x.max() + 2), np.arange(y.min(), y.max() + 2)],
              cmin=1, norm=plt.matplotlib.colors.LogNorm(), cmap="viridis")
fig.colorbar(h[3], ax=ax, label="observations per km²")
ax.set_title("Observations per 1 km cell"); ax.set_aspect("equal"); ax.set_axis_off()

## 5. Quality

- `quality_grade`: how many were confirmed by the community.
- `coordinates_obscured`: locations moved by up to several km
  (useless for 10 m pixels).
- `positional_accuracy`: how precise the location is, when the
  observer gave it.
- Species: different *Acacia* species may look different from space.

In [ ]:
print(inat["quality_grade"].value_counts().to_dict())
print("coordinates obscured:", int(inat["coordinates_obscured"].sum()))
acc = inat["positional_accuracy"]
print(f"positional_accuracy missing: {acc.isna().mean():.1%}")
pd.cut(acc.dropna(), [0, 10, 30, 100, 1000, np.inf], include_lowest=True,
       labels=["<=10 m", "10-30 m", "30-100 m", "100 m-1 km", ">1 km"]).value_counts().sort_index()

In [ ]:
inat["taxon_name"].value_counts().to_frame("n")

## 6. Observers

One very active observer imprints their own routes on the data: the
map can show where they walked rather than where *Acacia* is (RQ3).

In [ ]:
per_user = inat["user_login"].value_counts()
print(f"{len(per_user)} observers; the top one made {per_user.iloc[0] / len(inat):.1%} of the observations")
inat["observer"] = np.where(inat["user_login"] == per_user.index[0], "top observer", "others")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
pd.crosstab(year, inat["observer"]).loc[2015:].plot.bar(stacked=True, ax=axes[0], title="Observations per year")
for grp, colour in [("others", "tab:blue"), ("top observer", "tab:orange")]:
    inat[inat["observer"] == grp].plot(ax=axes[1], markersize=2, alpha=0.5, color=colour, label=grp)
axes[1].legend(markerscale=5); axes[1].set_title("Where"); axes[1].set_axis_off()
plt.tight_layout()

## Summary

- **7 943 observations**, 1992–2026, from 309 observers. Presence
  only: no absences.
- **When:** very uneven. 2021 alone holds 42.9%; 2024 15.4%; 2023
  only 3.5%; almost nothing before 2019. By month, March is the peak
  (16.4%, *A. dealbata* flowering), but every month has records.
- **One observer made 83.2%** of the observations, and almost all of
  the 2021 peak.
- **Same place:** 37.2% share a 10 m pixel with another observation.
- **Clustered:** 50 of 756 occupied 1 km cells hold 66.2% of the
  observations.
- **Quality:** 5 236 `research`, 2 449 `casual`, 258 `needs_id`. 43
  have obscured coordinates. `positional_accuracy` is missing for
  79.8%; where given, most are within 10 m.
- **Species:** *A. dealbata* 5 254, *A. melanoxylon* 1 898,
  *A. longifolia* 517, the rest few; 36 only at genus or section
  level.